# ML-09 — Validation and Research Claim Audit

This notebook does two things in the spirit of this week's session. First it reads two findings from FlyRank's March 2026 research paper the way an ML engineer reads a paper — naming the methodology question each one raises. Then it turns the same lens on my own Week-5 model: re-runs it under a more honest split with a before/after, hunts leakage, and rewrites my own overreaching sentences in safe claim language.

The tone for the paper section is **constructive, not a grade**: the paper discloses its own standards and limitations, and the point is to practice the same rigor on my own work. All language below stays public-safe: *observed, measured, directional, decision-support*.


## 1. Two paper findings + my methodology questions

**Finding A — the "Freshness Multiplier" (paper Finding #4).** The paper reports that 365+ day content refreshed within the last 30 days shows a 3.2x health-score boost and ~57x more impressions than untouched mature pages, and calls refresh timing "one of the strongest measured levers available."

- *Where does the outcome come from?* The health score is a FlyRank composite (impressions + position + CTR + scroll depth), and both it and impressions are read from the **same rolling 90-day snapshot** that also defines the "refreshed within 30 days" treatment flag.
- *My methodology question:* Does a single-snapshot comparison of refreshed vs untouched pages support a **causal** "refresh causes the lift" claim, or is it a selection effect — the pages chosen for refresh were already the ones most worth refreshing? A before/after on the *same* pages (or a matched cohort) would separate the treatment from the selection.

**Finding B — the "Anatomy of Growing Content" (paper Finding #1).** Growing pages are reported as 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days) than declining pages.

- *Where does the label come from?* "Growing" vs "declining" is the 30-day impression trend (up vs down) — the same window that also produces the profile metrics (impressions, position) being compared.
- *My methodology question:* The two claims are entangled: growing pages are **both longer and younger**, and the paper's own correlation appendix shows content age and word count move together (r ≈ −0.52). Does "longer = growing" survive an age-controlled comparison, or is length partly standing in for age?


In [1]:
# The two findings and their questions, as a table
import pandas as pd

findings = pd.DataFrame([
    {
        "Finding": "Freshness Multiplier (#4)",
        "Paper claim": "Refreshing 365+ day pages gives 3.2x health / ~57x impressions",
        "Outcome origin": "Health score + impressions, same 90-day snapshot as the 'refreshed <30d' flag",
        "Methodology question": "Cross-sectional vs before/after: is the lift causal, or is it selection of which pages got refreshed?",
    },
    {
        "Finding": "Anatomy of Growing Content (#1)",
        "Paper claim": "Growing pages are 37.6% longer and 20% younger than declining pages",
        "Outcome origin": "'Growing/declining' = 30-day impression trend; profile metrics from the same window",
        "Methodology question": "Is 'length predicts growth' robust after controlling for age, given age and word count are entangled?",
    },
])
findings


,Finding,Paper claim,Outcome origin,Methodology question
0,Freshness Multiplier (#4),Refreshing 365+ day pages gives 3.2x health / ...,"Health score + impressions, same 90-day snapsh...",Cross-sectional vs before/after: is the lift c...
1,Anatomy of Growing Content (#1),Growing pages are 37.6% longer and 20% younger...,'Growing/declining' = 30-day impression trend;...,Is 'length predicts growth' robust after contr...


## 2. My model under an honest split (before/after)

My Week-5 model already used a client-holdout split. To show *why* that matters, I re-run the identical feature matrix and models under **two splits** and put them side by side:

- **BEFORE (naive):** a random stratified row split (seed 42). Rows from the same client land in both train and test, so the model can memorize a client's template, traffic scale, and instrumentation quirks.
- **AFTER (honest):** the client-holdout split (seed 42, ~20% of clients held out) — the model must score pages from clients it never trained on.

The gap between the two columns is the amount of fake skill the random split was giving away. The cell below builds the Week-5 feature set exactly and trains all four Week-5 models under both splits.


In [2]:
# ---- Rebuild the Week-5 feature matrix (identical to w05_model.ipynb) ----
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

ROOT = Path(os.getcwd())
while not (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
df["declining"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

NUMERIC = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

num = df[NUMERIC].apply(pd.to_numeric, errors="coerce")
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "users_90d",
            "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "search_volume"]:
    num[f"log_{col}"] = np.log1p(num[col])
num["has_position"] = (df["avg_position"] > 0).astype(int)
num["has_word_count"] = (df["word_count"] > 0).astype(int)
num = num.replace([np.inf, -np.inf], np.nan).fillna(0)

cat = df[CATEGORICAL].fillna("unknown").astype(str)
cat_enc = pd.get_dummies(cat, prefix=CATEGORICAL, dtype=float)
X = pd.concat([num.reset_index(drop=True), cat_enc.reset_index(drop=True)], axis=1)
y = df["declining"].to_numpy()

RANDOM_STATE = 42

def make_models():
    return {
        "logistic_regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
        ]),
        "decision_tree": DecisionTreeClassifier(
            max_depth=5, min_samples_leaf=50, class_weight="balanced", random_state=RANDOM_STATE),
        "random_forest": RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=25,
            class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
        "hist_gradient_boosting": HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=50,
            l2_regularization=1.0, random_state=RANDOM_STATE),
    }

def p_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(labels)[order[:k]].mean())

# BEFORE: random stratified row split
idx = np.arange(len(df))
tr_rand, te_rand = train_test_split(idx, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# AFTER: client-holdout split
rng = np.random.default_rng(RANDOM_STATE)
clients = df["client_id"].drop_duplicates().to_numpy()
shuffled = rng.permutation(clients)
n_test_clients = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test_clients])
test_mask = df["client_id"].isin(test_clients).to_numpy()
tr_hold, te_hold = np.where(~test_mask)[0], np.where(test_mask)[0]

print(f"random split:   test {len(te_rand):,} rows, positive rate {y[te_rand].mean():.3f}")
print(f"client holdout: test {len(te_hold):,} rows, positive rate {y[te_hold].mean():.3f}")

rows = []
for name, template in make_models().items():
    import copy
    m_rand = copy.deepcopy(template); m_hold = copy.deepcopy(template)
    m_rand.fit(X.iloc[tr_rand], y[tr_rand])
    m_hold.fit(X.iloc[tr_hold], y[tr_hold])
    p_rand = m_rand.predict_proba(X.iloc[te_rand])[:, 1]
    p_hold = m_hold.predict_proba(X.iloc[te_hold])[:, 1]
    rows.append({
        "model": name,
        "P@50 random split": round(p_at_k(p_rand, y[te_rand], 50), 3),
        "P@50 client holdout": round(p_at_k(p_hold, y[te_hold], 50), 3),
        "gap (fake skill)": round(p_at_k(p_rand, y[te_rand], 50) - p_at_k(p_hold, y[te_hold], 50), 3),
        "AUC random": round(roc_auc_score(y[te_rand], p_rand), 3),
        "AUC holdout": round(roc_auc_score(y[te_hold], p_hold), 3),
    })

before_after = pd.DataFrame(rows)
print(before_after.to_string(index=False))

# keep the honest-split arrays for the leakage section
X_train, X_test, y_train, y_test = X.iloc[tr_hold], X.iloc[te_hold], y[tr_hold], y[te_hold]

OUT_DIR = ROOT / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
receipt = {
    "split_improvement": before_after.to_dict(orient="records"),
    "random_split_test_rows": int(len(te_rand)),
    "client_holdout_test_rows": int(len(te_hold)),
    "random_split_test_base_rate": round(float(y[te_rand].mean()), 4),
    "client_holdout_test_base_rate": round(float(y[te_hold].mean()), 4),
}
(OUT_DIR / "w06_validation_metrics.json").write_text(json.dumps(receipt, indent=2))
print(f"\nwrote {OUT_DIR / 'w06_validation_metrics.json'}")


random split:   test 6,000 rows, positive rate 0.542
client holdout: test 2,325 rows, positive rate 0.391


                 model  P@50 random split  P@50 client holdout  gap (fake skill)  AUC random  AUC holdout
   logistic_regression               0.88                 0.72              0.16       0.723        0.711
         decision_tree               0.90                 0.56              0.34       0.699        0.735
         random_forest               0.90                 0.66              0.24       0.750        0.735
hist_gradient_boosting               0.94                 0.72              0.22       0.770        0.721

wrote /Users/egealgel/Documents/FlyRankAI/flyrank-ml-internship-starter/work/outputs/w06_validation_metrics.json


**Reading the before/after.** Under the random split every model looks stronger than it is — Precision@50 is inflated by roughly 0.16–0.34 because pages from the same client leak across the split. Under the honest client-holdout the same models drop to their real numbers (the ones Week 5 reported). The gap *is* the finding: if I had validated with a random split, I would have shipped an inflated claim. Note also the base rate drops from 0.542 to 0.391 once unseen clients are scored — the honest split is genuinely harder.


## 3. Leakage audit

The same hunt from Week 3, now on the final Week-5 feature set. I test the two leak shapes the skill warns about: **label-derived** features (the trend column itself) and **same-window** counters (days of presence in the 90-day window). All tests run under the honest client-holdout split, so the inflation is measured on unseen clients.


In [3]:
# ---- Leak test 1: the label source itself (decisive) ----
X_plus_trend = X.copy()
X_plus_trend["trend_pct"] = df["trend_pct"].fillna(0).to_numpy()

lr = Pipeline([("scaler", StandardScaler()),
               ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
lr.fit(X_train, y_train)
honest_proba = lr.predict_proba(X_test)[:, 1]

lr_leak = Pipeline([("scaler", StandardScaler()),
                    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
lr_leak.fit(X_plus_trend.iloc[X_train.index], y_train)
leak_proba = lr_leak.predict_proba(X_plus_trend.iloc[X_test.index])[:, 1]

print("honest features          : P@50 =", round(p_at_k(honest_proba, y_test, 50), 3),
      " AUC =", round(roc_auc_score(y_test, honest_proba), 3))
print("+ trend_pct (label source): P@50 =", round(p_at_k(leak_proba, y_test, 50), 3),
      " AUC =", round(roc_auc_score(y_test, leak_proba), 3))

# ---- Leak test 2: a same-window presence counter (subtle) ----
# HistGradientBoosting on ONLY days_with_impressions reaches almost the full model's AUC,
# because that counter nearly encodes the trend-label categories (flat vs down/up/stable).
single = pd.DataFrame({"days_with_impressions": df["days_with_impressions"].to_numpy()})
hgb = HistGradientBoostingClassifier(max_iter=200, random_state=RANDOM_STATE)
hgb.fit(single.iloc[X_train.index], y_train)
single_proba = hgb.predict_proba(single.iloc[X_test.index])[:, 1]
print("days_with_impressions ONLY: P@50 =", round(p_at_k(single_proba, y_test, 50), 3),
      " AUC =", round(roc_auc_score(y_test, single_proba), 3))

# ---- Feature audit: assert the final feature set is clean ----
final_cols = set(X.columns)
forbidden = {
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
    "days_with_impressions", "days_with_sessions",
    "content_id", "client_id", "provider_used", "model_used",
}
overlap = forbidden & final_cols
print("\nforbidden columns present in final features:", sorted(overlap) if overlap else "none")
assert not overlap, "leaky column found in the final feature set!"
print("final feature count:", X.shape[1])


honest features          : P@50 = 0.72  AUC = 0.711
+ trend_pct (label source): P@50 = 1.0  AUC = 0.999


days_with_impressions ONLY: P@50 = 0.74  AUC = 0.758

forbidden columns present in final features: none
final feature count: 63


**Leakage audit, read honestly.**

- **The label source is lethal and caught:** adding `trend_pct` (the exact column the label is computed from) sends Precision@50 to 1.000 and AUC to 0.999. That is the "model quietly reads the answer" signature, and the audit confirms it is excluded.
- **The subtle one is also excluded:** `days_with_impressions` alone reaches AUC 0.758 on held-out clients with a single column — *above* the honest model's own AUC (0.711) — because it encodes the trend-label categories (few impression-days ≈ `flat`; many ≈ `down/up/stable`). It stays out of the final feature set.
- **The final feature set is clean:** no `trend_direction`, no `trend_pct`, no `*_last_30d`/`*_prev_30d` windows, no IDs, no provider/model metadata, no product flags.

**One limitation stays visible:** the starter label itself is defined on the current trailing window, so every 90-day feature overlaps the label window. This audit removes the *direct* leaks; the residual overlap is inherent to the starter slice and can only be resolved with the warehouse's forward window — the Week-6+ job.


## 4. Claim rewrite

The boldest sentence style in my Week-5 write-up was the one that implied the model knows the future. Here is the before, the problem, and the rewritten version in safe language.


In [4]:
claims = pd.DataFrame([
    {
        "Version": "BEFORE (overreaches)",
        "Sentence": "The model identifies which pages will decline, so reviewers should trust its top-50 picks.",
    },
    {
        "Version": "AFTER (safe)",
        "Sentence": ("On held-out clients, the learned model's top-50 list contains a measured 72% of pages "
                     "the current 90-day snapshot already labels declining, versus 26% for the Week-4 rule — "
                     "directional, decision-support evidence that the learned ranking surfaces more of the pages "
                     "the snapshot flags, not a prediction of future decline."),
    },
])
claims


,Version,Sentence
0,BEFORE (overreaches),"The model identifies which pages will decline,..."
1,AFTER (safe),"On held-out clients, the learned model's top-5..."


The same tightening applies to the Week-5 summary line: *"every learned model beats it by a wide margin"* becomes *"in this snapshot and on this client-holdout split, each learned model's Precision@50 exceeds the Week-4 rule's, with the largest gap for logistic regression and gradient boosting (0.72 vs 0.26)."* The numbers are identical; the claim no longer pretends to generalise beyond what was measured.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed via nbconvert)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.
